<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_14_Putting_It_All_Together.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [2]:
import pandas as pd

# Replace with the path to your downloaded file
transactions_2 = "/content/drive/MyDrive/data_files/transactions_2.csv"

transactions_df = pd.read_csv(transactions_2)
print(transactions_df)

        user_id      amount     transaction_date     status
0           4.0  367.154980  2025-01-01 00:00:00  completed
1           2.0   42.518694  2025-01-01 00:01:00     failed
2           3.0  204.574847  2025-01-01 00:02:00     failed
3           3.0  315.463172  2025-01-01 00:03:00  completed
4           2.0  346.918579  2025-01-01 00:04:00     failed
...         ...         ...                  ...        ...
499995      4.0  236.209566  2025-12-14 05:15:00    pending
499996      3.0  241.635235  2025-12-14 05:16:00  completed
499997      2.0  293.924346  2025-12-14 05:17:00  completed
499998      4.0    7.685396  2025-12-14 05:18:00     failed
499999      NaN  140.699505  2025-12-14 05:19:00  completed

[500000 rows x 4 columns]


In [3]:
import pandas as pd
import requests
import sqlite3

def process_transactions(input_path: str,  my_list: list):
    # Load data
    df = pd.read_csv(input_path)

    # Mixed cleaning and transformation
    df = df.dropna(subset=['user_id', 'amount'])
    df = df[df['amount'] > 0]

    # Enrichment with API call inside loop
    name_map = {}
    for user_id in my_list:
        resp = requests.get(f"https://jsonplaceholder.typicode.com/users/{user_id}")
        resp = resp.json()
        name_map[resp['id']]=resp['name']
        df['user_name'] = df['user_id'].map(name_map).fillna('Unknown')

    # Summary
    summary = df.groupby(['user_name']).agg({
        'amount': 'sum',
        'status': 'count'
    }).rename(columns={'status': 'transactions_count'})

    # Write to database
    conn = sqlite3.connect("warehouse.db")
    columns=['user_name', 'amount', 'transactions_count']
    summary.to_sql("daily_summary", conn, if_exists='replace', index=False)
    conn.close()
    return summary

 # Replace with the path to your downloaded file
transactions = "/content/drive/MyDrive/data_files/transactions_2.csv"
user_id = [1, 2, 3, 4, 5, 6, 7, 8]
process_transactions(transactions, user_id)

,amount,transactions_count
user_name,,
Clementine Bauch,2.287673e+07,91066
Ervin Howell,2.283688e+07,90957
Leanne Graham,2.275122e+07,90971
Patricia Lebsack,2.268003e+07,90675


In [4]:
%pip install pandera

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.6/604.6 kB 13.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandera]



# **Centralized Error Handling**



In [5]:
import functools
import logging

logger = logging.getLogger(__name__)

def pipeline_step(func):
    """Log individual step transitions and centralize exception capturing."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Mark the exact moment the step begins
        logger.info(f"Starting step: {func.__name__}")
        try:
            result = func(*args, **kwargs)
            # Confirm clean completion
            logger.info(f"Finished step: {func.__name__}")
            return result

        except Exception:
            # Global Handler to record critical infrastructure failures
            logger.exception(f"Pipeline step failed: {func.__name__}")
            raise

    return wrapper

In [6]:
from contextlib import contextmanager
from typing import Iterator, TextIO
import os

@contextmanager
def create_file_stream(input_path: str) -> Iterator[TextIO]:
    """Open and yield a readable file stream."""
    if not os.path.exists(input_path):
        raise FileNotFoundError(
            f"Source file not found at: {input_path}"
        )

    with open(input_path, mode="r", encoding="utf-8") as file:
        yield file

## **Full Function (Refactored version)**

In [7]:
import requests
import pandas as pd
from typing import Dict, Iterator, List, TextIO
import functools
import pandera.pandas as pa
from pandera.typing import DataFrame, Series
from contextlib import contextmanager
import os

@contextmanager
def create_file_stream(input_path: str) -> Iterator[TextIO]:
    """Open and yield a readable file stream."""
    if not os.path.exists(input_path):
        raise FileNotFoundError(
            f"Source file not found at: {input_path}"
        )

    with open(input_path, mode="r", encoding="utf-8") as file:
        yield file

def load_data(input_path: str) -> pd.DataFrame:
    """Load CSV data using the file connector."""
    with create_file_stream(input_path) as file:
        return pd.read_csv(file)

def fetch_users_from_api(user_list: list[int]) -> list[dict]:
    """Fetch raw user records from the API."""
    users = []

    for user_id in user_list:
        try:
            response = requests.get(
                f"https://jsonplaceholder.typicode.com/users/{user_id}",timeout=5)

            response.raise_for_status()
            users.append(response.json())

        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 404:
                logger.warning("User %s not found (404). Skipping.", user_id)
                continue
            raise

    return users

def build_user_map(users: list[dict]) -> dict[int, str]:
    """Build a user ID to name lookup from API records."""
    return {
        user["id"]: user["name"]
        for user in users
    }

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Remove invalid rows (nulls and non-positive amounts)."""
    result = df.copy()
    result = result.dropna(subset=['user_id', 'amount'])
    result = result[result['amount'] > 0]
    return result

def enrich_data(df: pd.DataFrame, name_map: Dict[int,str]) -> pd.DataFrame:
    """Pure: Add user names using pre-fetched map."""
    result = df.copy()
    result['user_name'] = result['user_id'].map(name_map).fillna('Unknown')
    return result

class DailySummarySchema(pa.DataFrameModel):
    """Schema for the summarized transactions."""
    user_name: Series[str]
    amount: Series[float] = pa.Field(ge=0)
    transactions_count: Series[int] = pa.Field(ge=0)

@pa.check_output(DailySummarySchema)
def summarize_data(df: pd.DataFrame) -> pd.DataFrame[DailySummarySchema]:
    """Aggregate transactions per user."""
    summary = (df.groupby('user_name', as_index=False)
             .agg(amount=("amount", "sum"),
                  transactions_count=("status", "count")
                )
             )
    return summary

def save_to_database(df: pd.DataFrame, db_path: str = "warehouse.db"
) -> None:
    """Save summarized data to the database."""
    with sqlite3.connect(db_path) as conn:
        df.to_sql("daily_summary", conn, if_exists="replace",
            index=False
        )

# Error handling Decorator
def pipeline_step(func):
    """Logging and centralized error reporting."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"Starting step: {func.__name__}")
        try:
            result = func(*args, **kwargs)
            logger.info(f"Finished step: {func.__name__}")
            return result
        except Exception:
            logger.exception(f"Pipeline step failed: {func.__name__}")
            raise
    return wrapper

# Orchestrator
@pipeline_step
def run_pipeline(input_path: str, user_ids: list[int]) -> pd.DataFrame:
    """Coordinate the loading, transformation, and saving phases of the pipeline."""

    # 1. External System Side Effects (Impure Extraction)
    raw_data = load_data(input_path)
    raw_users = fetch_users_from_api(user_ids)

    # 2. Preparation & Structuring (Pure Translation)
    user_map = build_user_map(raw_users)

    # 3. Functional Transformations (Pure Business Logic)
    cleaned_data = clean_data(raw_data)
    enriched_data = enrich_data(cleaned_data, user_map)
    summary_data = summarize_data(enriched_data)

    # 4. Persistence Side Effects (Impure Storage)
    save_to_database(summary_data)

    return summary_data


# Replace with the path to your downloaded file
transactions = "/content/drive/MyDrive/data_files/transactions_2.csv"

if __name__ == "__main__":
    logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
    )
    # List of user IDs
    user_ids = [1, 2, 3, 4, 5, 6, 7, 8]
    try:
        results = run_pipeline(transactions, user_ids)
        print(results)
    except Exception:
         logger.error("Pipeline failed safely at runtime: %s", e)

2026-11-18 16:46:22,803 - INFO - Starting step: run_pipeline
2026-11-18 16:46:28,476 - INFO - Finished step: run_pipeline


          user_name        amount  transactions_count
0  Clementine Bauch  2.287673e+07               91066
1      Ervin Howell  2.283688e+07               90957
2     Leanne Graham  2.275122e+07               90971
3  Patricia Lebsack  2.268003e+07               90675


In [8]:
# conn = sqlite3.connect("warehouse.db")
# cursor = conn.cursor()

# cursor.execute("SELECT * FROM daily_summary")
# cursor.fetchall()

In [9]:
# cursor.execute("DROP TABLE daily_summary")

# **Full Optimized pipeline**

In [10]:
import logging
import sqlite3
import functools
from typing import Dict, List, Iterator, TextIO
import pandas as pd
import requests
import pandera.pandas as pa
from pandera.typing import DataFrame, Series
from contextlib import contextmanager
from tenacity import retry, stop_after_attempt, wait_fixed, retry_if_exception_type
import os

logger = logging.getLogger(__name__)

@contextmanager
def create_file_stream(input_path: str) -> Iterator[TextIO]:
    """Open and yield a readable file stream."""
    if not os.path.exists(input_path):
        raise FileNotFoundError(
            f"Source file not found at: {input_path}"
        )

    with open(input_path, mode="r", encoding="utf-8") as file:
        yield file

def load_data(input_path: str) -> pd.DataFrame:
    """Load CSV data using the file connector."""
    with create_file_stream(input_path) as file:
        return pd.read_csv(file)

# Optimize Network Reliability with Automated Retries
@retry(
    stop=stop_after_attempt(3),
    wait=wait_fixed(2),
    retry=retry_if_exception_type(requests.exceptions.RequestException)
)
def _execute_api_call(user_id: int) -> Dict:
    """Execute raw HTTP request with built-in retry safety rules."""
    response = requests.get(
        f"https://jsonplaceholder.typicode.com/users/{user_id}", timeout=5)

    response.raise_for_status()
    return response.json()

# Optimize Network Volume with an In-Memory Lookup Cache
@functools.lru_cache(maxsize=500)
def fetch_user_profile_cached(user_id: int) -> Dict:
    """Caching results to eliminate duplicate network requests."""
    return _execute_api_call(user_id)

def fetch_users_from_api(user_list: List[int]) -> List[Dict]:
    """Fetch user records efficiently by deduplicating input IDs."""
    users = []

    # Remove duplicates while preserving the original order
    for user_id in dict.fromkeys(user_list):
        try:
            user_data = fetch_user_profile_cached(user_id)
            users.append(user_data)
        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 404:
                logger.warning("User %s not found (404). Skipping.", user_id)
                continue
            raise
    return users

def build_user_map(users: List[Dict]) -> Dict[int, str]:
    """Build a user ID to name lookup from API records (Pure)."""
    return {user["id"]: user["name"] for user in users}

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean rows containing nulls or non-positive amounts (Pure)."""
    result = df.copy()
    result = result.dropna(subset=['user_id', 'amount'])
    result = result[result['amount'] > 0]
    return result

def enrich_data(df: pd.DataFrame, name_map: Dict[int, str]) -> pd.DataFrame:
    """Map names downstream, safely noting missing keys as Unknown (Pure)."""
    result = df.copy()
    result['user_name'] = result['user_id'].map(name_map).fillna('Unknown')
    return result

class DailySummarySchema(pa.DataFrameModel):
    """Schema requirements for finalized database aggregates."""
    user_name: Series[str]
    amount: Series[float] = pa.Field(ge=0)
    transactions_count: Series[int] = pa.Field(ge=0)

@pa.check_output(DailySummarySchema)
def summarize_data(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate transactional metrics per individual user name (Pure)."""
    summary = (df.groupby('user_name', as_index=False)
               .agg(amount=("amount", "sum"),
                    transactions_count=("status", "count")
                   )
              )
    return summary

# Optimize Memory Footprint with Generator Batches
def process_in_batches(df: pd.DataFrame,
                       batch_size: int = 1000) -> Iterator[pd.DataFrame]:
    """Yield DataFrame chunks to process data without overwhelming system memory."""
    for i in range(0, len(df), batch_size):
        yield df.iloc[i:i + batch_size]

def save_to_database(df: pd.DataFrame, db_path: str = "warehouse.db") -> None:
    """Save summarized data permanently to the SQLite database (Impure)."""
    with sqlite3.connect(db_path) as conn:
        df.to_sql("daily_summary", conn, if_exists="replace", index=False)

def pipeline_step(func):
    """Central operational error wrapper."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"Starting step: {func.__name__}")
        try:
            result = func(*args, **kwargs)
            logger.info(f"Finished step: {func.__name__}")
            return result
        except Exception:
            logger.exception(f"Pipeline step failed: {func.__name__}")
            raise
    return wrapper

@pipeline_step
def run_pipeline(input_path: str, user_ids: List[int]) -> pd.DataFrame:
    """Coordinate the loading, optimized transformation, and saving phases."""

    # Extract raw sets across network limits (Impure)
    raw_data = load_data(input_path)
    raw_users = fetch_users_from_api(user_ids)

    # Map user data (Pure)
    user_map = build_user_map(raw_users)

    intermediate_summaries = []

    # Stream data chunks to keep RAM metrics stable
    for batch in process_in_batches(raw_data, batch_size=1000):
        cleaned = clean_data(batch)
        enriched = enrich_data(cleaned, user_map)
        intermediate_summaries.append(summarize_data(enriched))

    final_summary = (
        pd.concat(intermediate_summaries)
        .groupby("user_name", as_index=False)
        .agg(
            amount=("amount", "sum"),
            transactions_count=("transactions_count", "sum")
        )
    )

    save_to_database(final_summary)
    return final_summary


# Replace with path to your downloaded file
transactions = "/content/drive/MyDrive/data_files/transactions_2.csv"

if __name__ == "__main__":
    logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
    )
    user_ids = [1, 2, 3, 4, 5, 6, 7, 8]
    try:
        results = run_pipeline(transactions, user_ids)
        print(results)
    except Exception as e:
         logger.error("Pipeline failed safely at runtime: %s", e)

2026-11-18 16:46:28,577 - INFO - Starting step: run_pipeline
2026-11-18 16:46:35,845 - INFO - Finished step: run_pipeline


          user_name        amount  transactions_count
0  Clementine Bauch  2.287673e+07               91066
1      Ervin Howell  2.283688e+07               90957
2     Leanne Graham  2.275122e+07               90971
3  Patricia Lebsack  2.268003e+07               90675


In [11]:
# cursor.execute("DROP TABLE daily_summary")